In [8]:
from openai import OpenAI
from dotenv import load_dotenv

import os
import json

load_dotenv(override=True)

True

In [9]:
llm_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("OPEN_ROUTER_KEY"),
)

MODEL = "openai/gpt-oss-120b"       # chat model (free, NVIDIA)

In [10]:
menu = [
    {"name": "Zinger Burger", "price": 850},
    {"name": "Chicken Biryani", "price": 450},
    {"name": "Club Sandwich", "price": 700}
]

complaint_list = []

order_list = [
    "OR-5555", "OR-6666", "OR-7777", "OR-8888", "OR-9999"
]

In [11]:
def fetch_menu():
    return menu

def log_complaint(customer_name, order_id, phone, complaint):
    temp_dict = {
        "customer_name": customer_name,
        "order_id": order_id,
        "phone": phone,
        "complaint": complaint,
    }
    complaint_list.append(temp_dict)
    return f'Total Complaint: {len(complaint_list)}'

def validate_order(order_id):
    if order_id in order_list:
        return True
    else:
        return False

tools = [
    {
        "type": "function",
        "function": {
                "name": "fetch_menu",
                "description": "Get the restaurant menu",
                # no inputs
                "parameters": {"type": "object", "properties": {}},
        },
    },
    {
    "type": "function",
    "function": {
        "strict": True,
        "name": "log_complaint",
        "description": "Log the complaint",
        "parameters": {
            "type": "object",
            "properties": {
                "customer_name": {"type": "string", "description": "Customer Name who is logging complaint."},
                "order_id": {"type": "string", "description": "Order ID like OR-0000"},
                "phone": {"type": "string", "description": "Customer Phone Number or '-'"},
                "complaint": {"type": "string", "description": "Customer complaint, not more than 8 words."},
            },
            "required": ["customer_name", "order_id", "phone", "complaint"],
            "additionalProperties": False,
        },
    },
},
    {
        "type": "function",
        "function": {
                "name": "validate_order",
                "description": "Validate the Order",
                # no inputs
                "parameters": {"type": "object", "properties": {
                    "order_id":{
                        "Type":"string",
                        "description":"The Order ID of which the cutomer ordered food, Order ID sgould be OR-0000"
                    }
                },
                "required": ["order_id"],
            "additionalProperties": False,
            },
        },
    },
]

In [12]:
messages = [
    {
      "role": "system",  
      "content": """
       You are the AI Customer Support Assistant for **Salman Pasta Restaurant**. Your job is to answer customer queries in a friendly, professional, and helpful way.

        ### Tools
        1. **`get_menu`** — Call whenever customers ask about menu items, prices, ingredients, or food recommendations. Use the tool's response to provide accurate information.
        2. **`log_complaint`** — Call whenever a customer raises a complaint. Collect the necessary details and log the complaint using this schema:

        ```python
        complaint = {
            "customer_name": "",
            "phone": "",
            "complaint": "",
            "order_id": "",
            "status": "Pending"
        }
        ```

        Ask for missing necessary details before logging. The `order_id` can be empty if unavailable. Set the initial status to `"Pending"`. Confirm the complaint is logged only after the tool succeeds.

        ### Guidelines
        - Keep replies short, natural, and friendly.
        - Never invent menu details, prices, or restaurant policies.
        - If you don't know something, politely say so.
        - Make every customer feel valued.
      """
    }
]


In [13]:
function_names = {
    "fetch_menu": fetch_menu,
    "log_complaint": log_complaint,
    "validate_order": validate_order,
}



def ask_assistant(question):

    messages.append({
        "role": "user", 
        "content": question
    })

    completion = llm_client.chat.completions.create(
        model=MODEL, 
        messages=messages, 
        tools=tools
    )
   
    reply = completion.choices[0].message

    print("LLM REPY", reply)

    if reply.tool_calls:                       # model asked for a tool
        messages.append(reply)

        for call in reply.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            result = function_names[call.function.name](**args)     # run our Python function
            print("Tool called:", call.function.name, args)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})

        completion = llm_client.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    return completion.choices[0].message.content

In [14]:
ask_assistant("Tell me the Menu?")

LLM REPY ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='fc_ea4c65a2-fb11-4d87-9b22-3d9e2e1512b1', function=Function(arguments='{}', name='fetch_menu'), type='function')], reasoning='We need to retrieve the menu using get_menu tool. The tool is named get_menu but in the schema it\'s fetch_menu. Probably we should call fetch_menu. The user asks "Tell me the Menu?" We\'ll call fetch_menu.')
Tool called: fetch_menu {}


'Here’s our current menu:\n\n| Item | Price (₹) |\n|------|-----------|\n| Zinger Burger | 850 |\n| Chicken Biryani | 450 |\n| Club Sandwich | 700 |\n\nLet me know if you’d like more details about any dish or have any other questions!'

In [15]:
ask_assistant("I ordered Burger but sandwich was served and the bill was made with burger, My name is Abdul Salam, OR-1778, phone is 03269222700")

LLM REPY ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='fc_4c1cdc7d-d37c-4c39-b24b-e1db4561d6cf', function=Function(arguments='{"complaint":"Received sandwich instead of burger","customer_name":"Abdul Salam","order_id":"OR-1778","phone":"03269222700"}', name='log_complaint'), type='function')], reasoning='User is complaining: ordered burger but got sandwich, bill was for burger. Need to log complaint. Gather details: name Abdul Salam, phone 03269222700, complaint brief (<=8 words). Need to phrase complaint short: "Received sandwich instead of burger". That\'s within 8 words (maybe 5). Order ID OR-1778. Use log_complaint tool.')
Tool called: log_complaint {'complaint': 'Received sandwich instead of burger', 'customer_name': 'Abdul Salam', 'order_id': 'OR-1778', 'phone': '03269222700'}


'Your complaint has been logged, Abdul. We’ll investigate the mix‑up and get back to you shortly. Thank you for letting us know!'

In [20]:
ask_assistant("My order ID is OR-1011, validate my order, is it a valid order id? Check from order_lists")

LLM REPY ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='fc_6939b0e0-577d-4b64-a725-07eca76a3423', function=Function(arguments='{"order_id":"OR-1011"}', name='validate_order'), type='function')], reasoning='Call validate_order with OR-1011.')
Tool called: validate_order {'order_id': 'OR-1011'}


'The order ID **OR‑1011** is not valid in our system. If you need further assistance or have any other questions, just let me know!'

In [17]:
print(complaint_list)

[{'customer_name': 'Abdul Salam', 'order_id': 'OR-1778', 'phone': '03269222700', 'complaint': 'Received sandwich instead of burger'}]


In [18]:
# ask_assistant("What is the restaurant's name")

In [19]:
# ask_assistant("Why is your restaurant name Salman Pasta?")